# Reproducible LLM relay experiments on lexigram

This notebook demonstrates the framework's built-in reproducibility path: a **seeded, config-driven** experiment over the Claude relay mapper, with OpenTelemetry tracing (`AITracer`) and structured metrics (`AIMetrics`) recorded on every conversion. No external experiment-tracking service is required — every run is pinned by a digest and persisted under `runs/<run_id>/`.


## The reproducibility contract

1. **Config-driven**: `experiment.yaml` holds every knob (seed, model, iterations, sampling, recording switches).
2. **Seeded**: all synthetic wire payloads, latencies, and token counts come from `random.Random(seed)` — a Python-standard PRNG, stable across runs and platforms.
3. **Digest-pinned**: `sha256(params + metrics + results)` — same seed, same digest.
4. **Checkpoints + traces**: per-iteration conversion checkpoints and OTel spans are written to `runs/<run_id>/`.


In [ ]:
from pathlib import Path

from harness import load_config, run_experiment

config = load_config(Path('experiment.yaml'))
config['experiment']


In [ ]:
out = Path('runs')
# Same seed twice, plus a different seed for contrast.
run_a = run_experiment(config, seed=42, out_dir=out)
run_b = run_experiment(config, seed=42, out_dir=out)
run_c = run_experiment(config, seed=7, out_dir=out)
print('run_a:', run_a.run_id)
print('run_b:', run_b.run_id)
print('run_c:', run_c.run_id)


In [ ]:
# Both same-seed runs must produce an identical digest...
assert run_a.digest == run_b.digest, 'same seed diverged!'
# ...and a different seed must not.
assert run_a.digest != run_c.digest, 'different seed collided!'
print('reproducibility: OK — same seed == same digest')
print('digest:', run_a.digest)


### What gets tracked per run

`AIMetrics` (from `lexigram-ai-observability`) records LLM request counts, token totals, latencies, and cost; `AITracer` emits OpenTelemetry spans. Both are dumped deterministically so a run can be audited or compared later.


In [ ]:
print('--- counters ---')
for name, series in run_a.metrics['counters'].items():
    print(f'{name}: {sum(series.values())}')
print('--- histograms ---')
for name, series in run_a.metrics['histograms'].items():
    print(f'{name}: {len(series['-'])} observations')
print('--- totals ---')
print(run_a.result['totals'])


### Ablation / error analysis

The harness ships a tiny ablation switch: drop `thinking` blocks from the wire payloads and compare metrics against the control run (`metrics_delta`).


In [ ]:
from harness import metrics_delta

ablated = run_experiment(config, seed=42, out_dir=out, ablate='thinking')
deltas = metrics_delta(run_a, ablated)
print('metric deltas when thinking blocks are ablated:')
print(deltas)


## Artifacts & replay

Every run is on disk under `runs/<run_id>/`:

- `params.json` — pinned config, seed, ablation, config fingerprint
- `metrics.json` — AIMetrics snapshot
- `trace.json` — OTel span list (name + attributes)
- `result.json` — per-iteration conversions + totals
- `checkpoints/iteration_XX.json` — per-step checkpoints
- `reproducibility.json` — run_id + digest

Replay from the CLI:

```bash
python run_experiment.py --seed 42
python run_experiment.py --seed 42 --ablate thinking
```
